# 02d — Microbiology features

Calculate culture timing and ESBL status around admission. Duplicate spells are reduced before assigning episode-level features.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from pathlib import Path
from icare_risk.utils.paths import get_path, get_data_root_from_schema
from icare_risk.utils.preprocessing import keep_first_spell

# Define data path
DATA_DIR = get_path('data/synthetic/10000p') # Manually defined path
DATA_DIR = get_data_root_from_schema()       # Paramter schema_path= / default_dir=      
DATA_PROC_DIR = DATA_DIR / 'processed' 
KEYS = ["SUBJECT", "SPELL_IDENTIFIER"]
print(f"Using data from: {DATA_DIR}")

Using data from: /app/data/synthetic/10000p


In [3]:
# --- OPTION A: Load using pandas ---
def load_from_pandas(DATA_DIR):
    df_episodes = pd.read_parquet(DATA_DIR / "icare_episodes_anon.parquet")
    df_episodes = keep_first_spell(df_episodes)
    df_episodes["ADMISSION_DATE"] = pd.to_datetime(df_episodes["ADMISSION_DATE"])
    df_episodes["DISCHARGE_DATE"] = pd.to_datetime(df_episodes["DISCHARGE_DATE"])

    df_microbiology = pd.read_csv(DATA_DIR / "icare_microbiology_anon.csv")

    return {
        'episodes': df_episodes,
        'microbiology': df_microbiology
    }

# Load
dfs = load_from_pandas(DATA_DIR)

# Assign
df_episodes = dfs['episodes']
df_microbiology = dfs['microbiology']

episodes: 10,000 rows -> 10,000 unique spells


In [ ]:
# --- OPTION B: Load using DuckDB ---
from icare_risk.clinphen.io.loaders import load_from_duckdb

# Configure
schema_cfg = 'configs/schema.yaml'
phenotype_cfg = 'configs/phenotypes.yaml'
domains = ['episodes', 'microbiology']

# Load
dfs = load_from_duckdb(
    schema_path=schema_cfg,
    phenotype_path=phenotype_cfg,
    domains_to_load=domains,
    custom_codes={}, skip_mapping=True, 
    case_sensitive=False, verbose=1
)

# Assign
df_episodes = keep_first_spell(dfs['episodes'])

In [ ]:
from icare_risk.clinphen.utils import (
    compute_time_to_closest_culture,
    assign_esbl_to_episodes,
)



episodes: 10,000 rows -> 10,000 unique spells


In [ ]:
# Libraries
from icare_risk.clinphen.utils import (
    compute_time_to_closest_culture,
    assign_esbl_to_episodes,
)

df_micro_timing = compute_time_to_closest_culture(
    df_episodes=df_episodes,
    microbiology_df=df_micro,
    pre_admission_days=14,
    post_discharge_days=7,
    round_decimals=0,
)


from icare_risk.utils.paths import get_path
LOOKUP_3GCR_PATH = get_path('configs/lookups/3GCR.csv')
LOOKUP_3GCS_PATH = get_path('configs/lookups/3GCS.csv')

df_microbiology = assign_esbl_to_episodes(
    df_episodes=df_micro_timing,
    df_microbiology=df_micro,
    path_3gcr=str(LOOKUP_3GCR_PATH),
    path_3gcs=str(LOOKUP_3GCS_PATH),
    pre_admission_days=14,
    post_discharge_days=7,
)
df_microbiology = keep_first_spell(df_microbiology, "microbiology features")
display(df_microbiology.head())


Computing test-level ESBL labels on the fly...
microbiology features: 10,000 rows -> 10,000 unique spells


,SUBJECT,SPELL_IDENTIFIER,ENCNTR_ID,EPISODE_IDENTIFIER,ADMISSION_DATE,ADMISSION_TIME,AGE_AT_ADMISSION,DISCHARGE_DATE,MAIN_SPECIALTY_CODE,MAIN_SPECIALTY_CODE_DESC,INDEX_OF_MULTIPLE_DEPRIVATION_DECILE,closest_culture_date,time_to_culture_days,time_to_culture_hours,esbl_status
0,10001,7867287,2998828,3749211,2023-02-12,00:52:00,64,2023-02-26,140,oral surgery,1,NaN,NaN,NaN,unknown
1,10002,6551735,8909338,9353593,2022-11-16,23:43:00,72,2022-12-01,320,cardiology,1,2022-11-26,9.0,216.0,unknown
2,10003,4441228,1307783,3518105,2022-05-12,20:10:00,29,2022-05-27,320,cardiology,unknown,2022-05-25,12.0,292.0,unknown
3,10004,5593202,5695692,6593442,2023-10-11,00:52:00,19,2023-10-30,300,general medicine,4,NaN,NaN,NaN,unknown
4,10005,6532182,7455884,6203572,2023-08-14,05:42:00,50,2023-08-18,300,general medicine,7,NaN,NaN,NaN,unknown


In [10]:
filename = "02e_derived_features_microbiology.parquet"
output_path = DATA_PROC_DIR / filename
df_microbiology.to_parquet(output_path, index=False)
print(f"Saved: {output_path}")

Saved: /app/data/synthetic/10000p/processed/02e_derived_features_microbiology.parquet
